# Chapter 39: regularized linear models

This Notebook displays saved historical constructed summaries from fixtures/late; it does not retrain, rerun the 500-repetition experiment or reconstruct missing prediction rows. Ridge uses $\|y-Xw\|^2+\alpha\|w\|^2$; Lasso uses $\|y-Xw\|^2/(2n)+\alpha\|w\|_1$; Elastic Net adds $\alpha\rho\|w\|_1+\alpha(1-\rho)\|w\|^2/2$. Their alpha scales differ.

Historical hand, bias–variance and collinearity values are displayed, along with two frozen images. A unique RUN_DIR is created, but no new result file is saved. 

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch39_regularized_models_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json
import numpy as np, pandas as pd
from IPython.display import display, Image
CANDIDATE = ROOT / 'fixtures/late'
FROZEN = json.loads((CANDIDATE / 'ch39/ch39_results.json').read_text())
AUDIT = json.loads((CANDIDATE / 'display_audit.json').read_text())
print('Accepted historical summary display; no training or Ch39 row reconstruction.')


## 1. Frozen hand calculation

Display the archived four estimators for $X^\top X=4I$ and z=(2,1), including the saved maximum discrepancy. This cell does not recompute those fits.

In [ ]:
display(pd.DataFrame(FROZEN['hand_example']['closed_form'], index=['w1','w2']).T)
print('Frozen max difference:', FROZEN['hand_example']['max_abs_diff'])

## 2. Frozen bias–variance comparison

Read the theoretical and simulated MSE table across alpha, with $\alpha^*=\sigma^2/w^2$ under the stated scalar design. The image and simulations are historical, not regenerated.

In [ ]:
display(pd.DataFrame({k: FROZEN['bias_variance'][k] for k in ['alphas','mse_theory','mse_sim']}))
display(Image(filename=str(CANDIDATE/'ch39/fig39_2_bias_variance.png')))
print('Historical simulations only, not rerun.')

## 3. Frozen collinearity experiment

Display the archived correlation-0.99, 500-repetition results; no repetitions execute in this cell.

In [ ]:
display(pd.DataFrame(FROZEN['collinearity']['results']).T)
print('Frozen500 repeats, not rerun.')

## 4. Six-model summary and unresolved pairing

Own-date IC means use each model's valid dates: Ridge 750, Lasso 441 and Elastic Net 624, against 750 candidate label dates. Common dates, paired means and row-level missingness causes could not be reconstructed. P6 is another experiment and cannot fill that gap. Compressed NW(4) is not four-trading-day HAC, so old t-statistics are not displayed. panel_case is not called.

In [ ]:
rows={}
for model, v in FROZEN['panel_case']['results'].items():
 p=v['prediction']; n=v['portfolio']['fee_0.001']
 rows[model]={'own_IC': p['ic_mean'], 'own_valid_dates':p.get('ic_dates',0), 'candidate_dates':p.get('n_dates',750), 'paired_dates':'not_evaluated', 'paired_mean':'not_evaluated', 't_display':'not_reported', 'net_Sharpe':n['sharpe'], 'net_total':n['total_return']}
display(pd.DataFrame(rows).T)
display(Image(filename=str(CANDIDATE/'ch39/fig39_3_display_revision.png')))
assert AUDIT['chapter39']['lasso']['paired_ridge_dates'] is None

### Complete answer 39-C03

A paired comparison requires each model's effective date set intersected with Ridge's set. Without frozen prediction rows it remains unrecomputed, not fabricated. For dates 0,2,3,5,8, compressed lag four spans eight trading days; it cannot be described as four-day HAC.